# String Questions 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

## Que1: Leading Manufacturer Sales

**Difficulty:** Easy

### Problem

CVS Health wants to compare drug sales across pharmaceutical manufacturers. Each row in `tms_pharma_in` is one drug, with the revenue it generated in `total_sales`. Report every manufacturer alongside the combined sales of all its drugs, expressed to the nearest million dollars as the text `$<n> million`. List the manufacturer with the highest combined sales first, breaking ties alphabetically by manufacturer name.

**Schema columns:** `tms_pharma_in.product_id`, `tms_pharma_in.units_sold`, `tms_pharma_in.total_sales`, `tms_pharma_in.cogs`, `tms_pharma_in.manufacturer`, `tms_pharma_in.drug`

**Output columns:** `manufacturer`, `sale`

### Examples

#### Example 1

**Input:**

**tms_pharma_in:**

| product_id | units_sold | total_sales | cogs | manufacturer | drug |
|-----------:|-----------:|------------:|-----:|--------------|------|
| 9413 | 23622 | 2041758.41 | 1373721.70 | Biogen | UP and UP |
| 9374 | 10293 | 452.54 | 208876.01 | Eli Lilly | Zyprexa |
| 5090 | 48425 | 2521023.73 | 2742445.90 | Eli Lilly | Dermasorb |
| 6177 | 02350 | 500101.61 | 419174.97 | Biogen | Varicose Relief |
| 1361 | 44814 | 1084258.00 | 1006447.73 | Biogen | Burkhart |

**Output:**

| manufacturer | sale |
|--------------|------|
| Biogen | $4 million |
| Eli Lilly | $3 million |

**Explanation:** Biogen's three drugs sum to 2041758.41 + 500101.61 + 1084258 = 3626118.02, which rounds to $4 million. Eli Lilly's two drugs sum to 293452.54 + 2521023.73 = 2814476.27, which rounds to $3 million.

### Constraints

- Combined sales are rounded to the nearest million and formatted as the text `$<n> million`.
- Order by combined sales from highest to lowest, then by manufacturer name alphabetically for ties.
- Return results matching the expected output schema and order.

In [0]:
tms_pharma_in_data = [(9413,23622,2041758.41,1373721.70,"Biogen","UP and UP"),(9374,10293,452.54,208876.01,"Eli Lilly","Zyprexa"),(5090,48425,2521023.73,2742445.90,"Eli Lilly","Dermasorb"),(6177,2350,500101.61,419174.97,"Biogen","Varicose Relief"),(1361,44814,1084258.00,1006447.73,"Biogen","Burkhart")]
tms_pharma_in_df = spark.createDataFrame(tms_pharma_in_data, ["product_id","units_sold","total_sales","cogs","manufacturer","drug"])

display(tms_pharma_in_df)

grouped_df = (
tms_pharma_in_df.groupBy("manufacturer").agg(
    sum("total_sales").alias("total_sales")
))

output_df = (
grouped_df
    .withColumn("sale", 
        concat(lit("$"), round(col("total_sales") / 1000000), lit(" Million"))
    )
    .orderBy("total_sales", "manufacturer")
    .drop(col("total_sales").desc())
)

display(output_df)



## Que2: String Operations and Pattern Matching

**Difficulty:** Medium

### Problem

Extract email domain, parse full names into first/last components, and detect phone numbers in bio text using regex pattern matching.

**Schema columns:** `user_profiles.user_id`, `user_profiles.full_name`, `user_profiles.email`, `user_profiles.bio`

**Output columns:** `user_id`, `first_name`, `last_name`, `email_domain`, `has_phone`

Sort by the first column in ascending order.

### Examples

#### Example 1

**Input:**

**user_profiles:**

| user_id | full_name | email | bio |
|--------:|-----------|-------|-----|
| 1 | John Smith | [email protected] | Software engineer from NYC. Call me at 555-123-4567 |
| 2 | Sarah Johnson | [email protected] | Marketing professional in San Francisco |
| 3 | Michael Lee | [email protected] | Product manager. Available at 555-987-6543 for consulting |
| 4 | Emily Davis | [email protected] | Designer and artist |
| 5 | Robert Brown | [email protected] | Data scientist. Contact: 555-234-5678 |

**Output:**

| user_id | first_name | last_name | email_domain | has_phone |
|--------:|-----------|----------|-------------|:---------:|
| 1 | John | Smith | gmail.com | 1 |
| 2 | Sarah | Johnson | yahoo.com | 0 |
| 3 | Michael | Lee | company.org | 1 |
| 4 | Emily | Davis | outlook.com | 0 |
| 5 | Robert | Brown | example.net | 1 |

**Explanation:** The output is derived by applying the required transformations to the input data.

### Constraints

- Phone pattern: exactly XXX-XXX-XXXX (3 digits - 3 digits - 4 digits).
- Order by `user_id` ASC.
- `has_phone` is binary (1 or 0).

In [0]:
user_profiles_data = [(1,"John Smith","abc@gmail.com","Software engineer from NYC. Call me at 555-123-4567"),(2,"Sarah Johnson","abc@abc.com","Marketing professional in San Francisco"),(3,"Michael Lee","abc@outlook.com","Product manager. Available at 555-987-6543 for consulting"),(4,"Emily Davis","abc@yahoo.com","Designer and artist"),(5,"Robert Brown","abc@example.nex","Data scientist. Contact: 555-234-5678")]
user_profiles_df = spark.createDataFrame(user_profiles_data, ["user_id","full_name","email","bio"])

display(user_profiles_df)


output_df = (
user_profiles_df
    .withColumn("full_name", split(col("full_name"), " "))
    .withColumn("first_name", col("full_name")[0])
    .withColumn("last_name", col("full_name")[1])
    .withColumn("email_domain", regexp_extract(col("email"), r"@(.+)$", 1))
    .withColumn("has_no",
        when(col("bio").rlike(r"\d{3}-\d{3}-\d{4}"), 1).otherwise(0)
    )

    .select("user_id", "first_name", "last_name", "email_domain", "has_no")
    .orderBy("user_id")

)

display(output_df)


## Que3: The PADS - Format Names with Occupations

**Difficulty:** Medium

### Problem

Given a directory of people and their occupations, produce a single-column report in two sections:
1. One line per person formatted as `Name(X)`, where X is the first letter of that person's occupation, listed alphabetically by name.
2. One summary line per occupation reading `There are N Ys.`, where N is how many people have that occupation and Y is the plural of the occupation name.

The summary lines come after all the name lines and are ordered by count from lowest to highest; occupations with the same count are listed alphabetically.

**Schema columns:** `occupations.name`, `occupations.occupation`

**Output columns:** `output_line`

### Examples

#### Example 1

**Input:**

**occupations:**

| name | occupation |
|------|------------|
| Alice | Doctor |
| Bob | Actor |
| Charlie | Singer |
| Eve | Doctor |
| Frank | Actor |

**Output:**

| output_line |
|-------------|
| Alice(D) |
| Bob(A) |
| Charlie(S) |
| Eve(D) |
| Frank(A) |
| There are 1 Singers. |
| There are 2 Actors. |
| There are 2 Doctors. |

**Explanation:** The five people sort alphabetically into the Name(X) lines. In the summary, Singer has 1 person so it comes first; Actor and Doctor each have 2 people, so they follow ordered alphabetically.

### Constraints

- Name lines: `Name(X)` where X is the first letter of the occupation, sorted alphabetically by name.
- Summary lines: `There are N Ys.` using the plural of the occupation, placed after all name lines.
- Order summary lines by count ascending, breaking ties by occupation name alphabetically.
- Everything is returned in a single output column.
- Return results matching the expected output schema and order.

In [0]:
occupations_data = [("Alice","Doctor"),("Bob","Actor"),("Charlie","Singer"),("Eve","Doctor"),("Frank","Actor")]
occupations_df = spark.createDataFrame(occupations_data, ["name","occupation"])

display(occupations_df)

summary_1_df = (
occupations_df
    .withColumn("output", concat(  col("name"), lit("("),    substring(col("occupation"), 1, 1),    lit(")")   )   )
    .orderBy("name")
    .select("output")
)

grouped_df = (
occupations_df.groupBy("occupation").agg(
    count(col("occupation")).alias("total_count")
)
.orderBy("total_count")
)


summary_2_df = (
grouped_df
    .withColumn("output", concat(  lit("There are "), col("total_count"), lit(" "), col("occupation"), lit("s.")   ))
    .select("output")
)


output_df = summary_1_df.union(summary_2_df)

display(output_df)


## Que4: Create Features from Text

**Difficulty:** Hard

### Problem

A product team needs deterministic text features for each review. Return its whitespace-delimited word count, total character count, `avg_word_length` as total character count divided by word count, whether any negative keyword token appears, and the count of positive keyword tokens minus negative keyword tokens.

**Schema columns:** `reviews.review_id`, `reviews.product_id`, `reviews.review_text`, `reviews.review_date`

**Output columns:** `review_id`, `product_id`, `word_count`, `char_count`, `avg_word_length`, `contains_negative`, `sentiment_score`

### Examples

#### Example 1

**Input:**

**reviews:**

| review_id | product_id | review_text | review_date |
|----------:|-----------|-------------|-------------|
| 1 | P002 | This product is great and amazing | 2024-01-02 |
| 2 | P003 | Terrible quality, very bad experience | 2024-01-03 |
| 5 | P001 | Excellent, best product I have | 2024-01-06 |

**Output:**

| review_id | product_id | word_count | char_count | avg_word_length | contains_negative | sentiment_score |
|----------:|-----------|----------:|----------:|----------------:|:-----------------:|----------------:|
| 1 | P002 | 6 | 33 | 5.50 | 0 | 2 |
| 2 | P003 | 5 | 37 | 7.40 | 1 | -2 |
| 5 | P001 | 5 | 30 | 6.00 | 0 | 1 |

**Explanation:** Review 1 has 33 total characters across six tokens, so its `avg_word_length` is 5.5.

### Constraints

- Split words only on whitespace and retain punctuation in tokens.
- Count every character in `review_text`, including whitespace and punctuation, in `char_count`.
- Compute `avg_word_length` = `char_count` / `word_count`, then round to two decimals.
- Match keywords case-insensitively against exact tokens: positive — `great`, `good`, `excellent`, `amazing`, `love`, `best`; negative — `bad`, `terrible`, `awful`, `worst`, `poor`, `hate`.
- Sort by `review_id` ascending.
- Return results matching the expected output schema and order.

In [0]:
reviews_data = [(1,"P002","               This product is great    and amazing    ","2024-01-02"),(2,"P003","   Terrible quality  , very bad experience","2024-01-03"),(5,"P001","Excellent   , best product I have","2024-01-06")]
reviews_df = spark.createDataFrame(reviews_data, ["review_id","product_id","review_text","review_date"])

reviews_df = reviews_df.withColumn("review_text", trim(lower(col("review_text"))))

display(reviews_df)

positive_list = {"great", "good", "excellent", "amazing", "love", "best"}
negative_list = {"bad", "terrible", "awful", "worst", "poor", "hate"}

@udf(returnType=StructType([
    StructField("word_count", IntegerType(), True),
    StructField("negative_count", IntegerType(), True),
    StructField("positive_count", IntegerType(), True)
]))
def count_words(s):
    s = s.split(" ")
    count = 0
    negative_count = 0
    positive_count = 0
    for word in s:
        if word.isalpha():
            count += 1
        if word in negative_list:
            negative_count += 1
        if word in positive_list:
            positive_count += 1
        
    return (count, negative_count, positive_count)



string_opp_df = (
    reviews_df
    .withColumn(
        "word_stats",
        count_words(col("review_text"))
    )
    .withColumn("word_count", col("word_stats.word_count"))
    .withColumn("negative_count", col("word_stats.negative_count"))
    .withColumn("positive_count", col("word_stats.positive_count"))
    .withColumn("char_count",length(regexp_replace(col("review_text"), " ", "")))
    .withColumn("avg_word_length", round(col("char_count")/col("word_count"), 2))
    .drop("word_stats")
)


display(string_opp_df)


## Que5: Sports Match Score Summary

**Difficulty:** Easy

### Problem

A local team is analyzing their match performance over a season. For every game played, they have stored the match date, whether they won or lost, and the score difference. Add a `Scoreline` column formatted as `Win by N` or `Loss by N`.

**Schema columns:** `mss_score.Date`, `mss_score.Result`, `mss_score.Differential`

**Output columns:** `Date`, `Result`, `Differential`, `Scoreline`

### Examples

#### Example 1

**Input:**

**mss_score:**

| Date | Result | Differential |
|------------|--------|-------------:|
| 2023-10-01 | Win | 10 |
| 2023-10-08 | Loss | 7 |
| 2023-10-15 | Win | 3 |

**Output:**

| Date | Result | Differential | Scoreline |
|------------|--------|-------------:|-----------|
| 2023-10-01 | Win | 10 | Win by 10 |
| 2023-10-08 | Loss | 7 | Loss by 7 |
| 2023-10-15 | Win | 3 | Win by 3 |

### Constraints

- Handle NULL values appropriately.
- Return results matching the expected output schema and order.

In [0]:
mss_score_data = [("2023-10-01","Win",10),("2023-10-08","Loss",7),("2023-10-15","Win",3)]
mss_score_df = spark.createDataFrame(mss_score_data, ["Date","Result","Differential"])

display(mss_score_df)

output_df = (
mss_score_df
    .withColumn("scoreline", concat(col("Result"), lit(" by "), col("Differential")))
)

display(output_df)

## Que6: Email Validation Filter

**Difficulty:** Easy

### Problem

Filter the signup list down to valid company-domain emails. You are a data engineer at Airbnb cleaning a partner signup list before a campaign send. The `fve_sample` table contains user records, but some email values are malformed or belong to outside domains. Only addresses on the `dataplatform.com` domain that follow the format rules below should be kept.

An email is valid only if:
- It starts with a letter (a-z or A-Z).
- The rest of the local part (before the @) contains only letters, digits, underscores (_), dots (.), or hyphens (-).
- It ends with exactly `@dataplatform.com` with nothing after it.

Return `customer_id`, `full_name`, and `email` for the matching rows, in ascending order of `customer_id`.

**Schema columns:** `fve_sample.customer_id`, `fve_sample.full_name`, `fve_sample.email`

**Output columns:** `customer_id`, `full_name`, `email`

### Examples

#### Example 1

**Input:**

**fve_sample:**

| customer_id | full_name | email |
|------------:|-----------|-------|
| 1 | Alice | [email protected] |
| 2 | Bob | bob_the_great |
| 3 | Charlie | [email protected] |
| 4 | Daniel | [email protected] |
| 5 | Eve | eve#[email protected] |
| 6 | Frank | [email protected] |

**Output:**

| customer_id | full_name | email |
|------------:|-----------|-------|
| 1 | Alice | [email protected] |
| 3 | Charlie | [email protected] |
| 4 | Daniel | [email protected] |

**Explanation:** Alice, Charlie, and Daniel all start with a letter, use only allowed characters, and end in `@dataplatform.com`. Bob has no @ or domain, Eve contains the disallowed #, and Frank is on gmail.com.

### Constraints

- The email must start with a letter (a-z or A-Z).
- The remaining local-part characters are limited to letters, digits, underscore, dot, and hyphen.
- The email must end with exactly `@dataplatform.com`, with no trailing characters.
- Emails with any other domain, a missing @, or a disallowed character are invalid and excluded.
- Output columns must be exactly `customer_id`, `full_name`, `email`.
- Return rows in ascending `customer_id` order.

In [0]:
fve_sample_data = [(1,"Alice","[email protected]"),(2,"Bob","bob_the_great"),(3,"Charlie","[email protected]"),(4,"Daniel","[email protected]"),(5,"Eve","eve#[email protected]"),(6,"Frank","[email protected]")]
fve_sample_df = spark.createDataFrame(fve_sample_data, ["customer_id","full_name","email"])
r = r"^[A-Za-z][A-Za-z0-9_.-]*@dataplatform\.com$"
display(fve_sample_df)

## Que7: Find Users With Valid E-Mails

**Difficulty:** Easy

### Problem

A user directory needs records whose email address follows a restricted format. A valid address starts with a letter, continues with zero or more letters, digits, periods, underscores, or dashes, then has @, a domain name, a period, and a letter-only suffix of at least two characters.

**Schema columns:** `users.user_id`, `users.name`, `users.mail`

**Output columns:** `user_id`, `name`, `mail`

### Examples

#### Example 1

**Input:**

**users:**

| user_id | name | mail |
|--------:|------|------|
| 1 | Alice | [email protected] |
| 2 | Bob | [email protected] |
| 3 | Charlie | [email protected] |
| 4 | David | [email protected] |
| 5 | Eve | [email protected] |

**Output:**

| user_id | name | mail |
|--------:|------|------|
| 1 | Alice | [email protected] |
| 2 | Bob | [email protected] |
| 3 | Charlie | [email protected] |
| 5 | Eve | [email protected] |

**Explanation:** David is excluded because his email prefix starts with a digit.

### Constraints

- The prefix must begin with an ASCII letter.
- After the first character, the prefix may contain letters, digits, `.`, `_`, or `-`.
- Between @ and the final period, allow one or more letters, digits, periods, or dashes.
- The final suffix must contain at least two letters.
- Order by `user_id` ascending.
- Return results matching the expected output schema and order.

In [0]:
users_data = [(1,"Alice","[email protected]"),(2,"Bob","[email protected]"),(3,"Charlie","[email protected]"),(4,"David","[email protected]"),(5,"Eve","[email protected]")]
users_df = spark.createDataFrame(users_data, ["user_id","name","mail"])

display(users_df)

r = r"^[A-Za-z][A-Za-z0-9_.-]*@dataplatform\.com$"

## Que8: Weather Station - Shortest and Longest City Names

**Difficulty:** Easy

### Problem

Query the two cities in STATION with shortest and longest CITY names and their respective lengths. If multiple cities share the same length, choose the one that comes first alphabetically. Return format: city_name length on separate lines.

**Schema columns:** `station.id`, `station.city`, `station.state`, `station.lat_n`, `station.long_w`

**Output columns:** `city`, `city_length`

Order the results by `LENGTH(city)`, `city`.

### Examples

#### Example 1

**Input:**

**station:**

| id | city | state | lat_n | long_w |
|---:|------|-------|------:|-------:|
| 1 | Boston | MA | 42.3601 | -71.0589 |
| 2 | Springfield | IL | 39.7817 | -89.6501 |
| 3 | Denver | CO | 39.7392 | -104.9903 |
| 4 | Miami | FL | 25.7617 | -80.1918 |
| 5 | Houston | TX | 29.7604 | -95.3698 |

**Output:**

| city | city_length |
|------|------------:|
| Miami | 5 |
| Springfield | 11 |

**Explanation:** From the station table, Miami has the shortest name (5) and Springfield has the longest (11).

### Constraints

- When multiple cities share the same shortest or longest name length, choose the one that comes first alphabetically.
- Return results matching the expected output schema and order.

In [0]:
station_data = [(1,"Boston","MA",42.3601,-71.0589),(2,"Springfield","IL",39.7817,-89.6501),(3,"Denver","CO",39.7392,-104.9903),(4,"Miami","FL",25.7617,-80.1918),(5,"Houston","TX",29.7604,-95.3698)]
station_df = spark.createDataFrame(station_data, ["id","city","state","lat_n","long_w"])

display(station_df)

station_df = (
station_df
    .withColumn("city_length", length(col("city")))
)

min_window = Window.orderBy(col("city_length"))
max_window = Window.orderBy(col("city_length").desc())

ranked_df = (
station_df
    .withColumn("min_rank", row_number().over(min_window))
    .withColumn("max_rank", row_number().over(max_window))
).filter((col("min_rank") == 1) | (col("max_rank") == 1)).select("city", "city_length").orderBy("city_length")


display(ranked_df)


## Que9: Draw The Triangle 1

**Difficulty:** Easy

### Problem

Given a configuration value `n`, generate a triangle where the first row has `n` asterisks, the second row has `n-1`, and so on until row `n` which has 1 asterisk. Each asterisk should be separated by a space.

**Schema columns:** `config.n`

**Output columns:** `row_num`, `stars`

Sort by the first column in ascending order.

### Examples

#### Example 1

**Input:**

**config:**

| n |
|--:|
| 5 |

**Output:**

| row_num | stars |
|--------:|-------|
| 1 | * * * * * |
| 2 | * * * * |
| 3 | * * * |
| 4 | * * |
| 5 | * |

**Explanation:** The output is derived by applying the required transformations to the input data.

### Constraints

- 1 ≤ n ≤ 100.
- Each asterisk is separated by a single space.
- Row numbers start from 1.
- Return results matching the expected output schema and order.

In [0]:
config_data = [(5,)]
config_df = spark.createDataFrame(config_data, ["n"])

display(config_df)

n = config_df.collect()[0][0]

new_df = spark.range(1, n + 1)

output_df = (
new_df
    .withColumnRenamed("id", "row_num")
    .withColumn(("starts"), repeat(lit("*"), (n + 1 - col("row_num"))))
)

display(output_df)


## Que10: Draw The Triangle 2

**Difficulty:** Easy

### Problem

Given a configuration value `n`, generate a triangle where the first row has 1 asterisk, the second row has 2, and so on until row `n` which has `n` asterisks. Each asterisk should be separated by a space.

**Schema columns:** `config.n`

**Output columns:** `row_num`, `stars`

Sort by the first column in ascending order.

### Examples

#### Example 1

**Input:**

**config:**

| n |
|--:|
| 5 |

**Output:**

| row_num | stars |
|--------:|-------|
| 1 | * |
| 2 | * * |
| 3 | * * * |
| 4 | * * * * |
| 5 | * * * * * |

**Explanation:** The output is derived by applying the required transformations to the input data.

### Constraints

- 1 ≤ n ≤ 100.
- Each asterisk is separated by a single space.
- Row numbers start from 1.
- Return results matching the expected output schema and order.

In [0]:
config_data = [(5,)]
config_df = spark.createDataFrame(config_data, ["n"])

display(config_df)

n = config_df.collect()[0][0]

new_df = spark.range(1, n + 1)

output_df = (
new_df
    .withColumnRenamed("id", "row_num")
    .withColumn(("starts"), repeat(lit("*"), col("row_num")))
)

display(output_df)


## Que11: Discount Calculation Error Detection

**Difficulty:** Medium

### Problem

Rebuild the discount column from the product description text. You are an analyst at a consumer goods company. Store teams type promotions directly into the product description as tags like `[10% off]`, and the separately maintained discount column has drifted out of sync in past loads.

Write a query that returns every row of `cgdc_sales` with all original columns, but with the `Discount` value recomputed from `Description`: if the description contains a tag of the form `[N% off]` (an integer percentage inside square brackets), the discount is N / 100 expressed as a decimal fraction; if no such tag is present, the discount is 0.0. Leave every other column unchanged and keep the rows in their original table order.

**Schema columns:** `cgdc_sales.storeid`, `cgdc_sales.productname`, `cgdc_sales.category`, `cgdc_sales.soldunits`, `cgdc_sales.description`, `cgdc_sales.discount`

**Output columns:** `StoreID`, `ProductName`, `Category`, `SoldUnits`, `Description`, `Discount`

### Examples

#### Example 1

**Input:**

**cgdc_sales:**

| storeid | productname | category | soldunits | description | discount |
|---------|------------|---------|----------:|-------------|--------:|
| S101 | Biscuits | Food | 120 | Tasty Biscuits [10% off] | 0.1 |
| S102 | Shampoo | Hygiene | 85 | Smoothens Hair [5% off] | 0.05 |
| S103 | Banana | Food | 150 | Fresh Bananas | 0 |
| S101 | Toothpaste | Hygiene | 300 | Protects Teeth | 0 |
| S102 | Shirt | Clothes | 65 | Cotton Shirts [20% off] | 0.2 |

**Output:**

| StoreID | ProductName | Category | SoldUnits | Description | Discount |
|---------|------------|---------|----------:|-------------|--------:|
| S101 | Biscuits | Food | 120 | Tasty Biscuits [10% off] | 0.1 |
| S102 | Shampoo | Hygiene | 85 | Smoothens Hair [5% off] | 0.05 |
| S103 | Banana | Food | 150 | Fresh Bananas | 0.0 |
| S101 | Toothpaste | Hygiene | 300 | Protects Teeth | 0.0 |
| S102 | Shirt | Clothes | 65 | Cotton Shirts [20% off] | 0.2 |

**Explanation:** "Tasty Biscuits [10% off]" contains the tag [10% off], so its recomputed discount is 10 / 100 = 0.1. "Fresh Bananas" has no tag, so its discount is 0.0.

### Constraints

- Return every row with all original columns.
- Recompute `Discount` by extracting the integer N from an `[N% off]` tag in `Description` and dividing by 100.
- `Discount` is 0.0 when no `[N% off]` tag is present.
- All other column values must pass through unchanged.
- Preserve the original table row order.
- Output columns must be exactly `StoreID`, `ProductName`, `Category`, `SoldUnits`, `Description`, `Discount`.

In [0]:
cgdc_sales_data = [("S101","Biscuits","Food",120,"Tasty Biscuits [10% off]",0.1),("S102","Shampoo","Hygiene",85,"Smoothens Hair [5% off]",0.05),("S103","Banana","Food",150,"Fresh Bananas",0.0),("S101","Toothpaste","Hygiene",300,"Protects Teeth",0.0),("S102","Shirt","Clothes",65,"Cotton Shirts [20% off]",0.2)]
cgdc_sales_df = spark.createDataFrame(cgdc_sales_data, ["storeid","productname","category","soldunits","description","discount"])

cgdc_sales_df = cgdc_sales_df.withColumn("discount_prct", regexp_extract(col("description"), r"\[(\d+)% off\]", 1).try_cast("int"))

display(cgdc_sales_df)

output_df = cgdc_sales_df.withColumn("discount",  when(col("discount_prct").isNotNull(), col("discount_prct")/100).otherwise(0))

display(output_df)


## Que12: Flooring Store Sales Analysis

**Difficulty:** Medium

### Problem

Assemble a clean order report by splitting packed name and product fields. You are a data analyst at the multinational flooring retailer Floors'R'Us. The sales system exports customer names as a single `full_name` string and product attributes packed into a comma-separated `product_info` string.

Write a query that joins `fr_orders` to `fr_customers` on `customer_id` and to `fr_products` on `product_id` (inner joins). Derive `first_name` and `last_name` by splitting `full_name` on the single space. Derive `product_type` and `product_color` by splitting `product_info` on the comma. Return the columns in exactly this order: `customer_id`, `first_name`, `last_name`, `location`, `order_id`, `product_type`, `product_color`, `product_id`, `quantity`, with rows in ascending order of `order_id`.

**Schema columns:** `fr_customers.customer_id`, `fr_customers.full_name`, `fr_customers.location`, `fr_orders.order_id`, `fr_orders.customer_id`, `fr_orders.product_id`, `fr_orders.quantity`, `fr_products.product_id`, `fr_products.product_info`

**Output columns:** `customer_id`, `first_name`, `last_name`, `location`, `order_id`, `product_type`, `product_color`, `product_id`, `quantity`

### Examples

#### Example 1

**Input:**

**fr_customers:**

| customer_id | full_name | location |
|------------:|-----------|----------|
| 1 | John Doe | Texas |
| 2 | Jane Smith | California |

**fr_orders:**

| order_id | customer_id | product_id | quantity |
|---------:|------------:|-----------:|---------:|
| 1001 | 1 | 101 | 5 |
| 1002 | 2 | 102 | 2 |

**fr_products:**

| product_id | product_info |
|-----------:|-------------|
| 101 | Carpet,Red |
| 102 | Tile,Blue |

**Output:**

| customer_id | first_name | last_name | location | order_id | product_type | product_color | product_id | quantity |
|------------:|-----------|----------|----------|--------:|-------------|--------------|----------:|---------:|
| 1 | John | Doe | Texas | 1001 | Carpet | Red | 101 | 5 |
| 2 | Jane | Smith | California | 1002 | Tile | Blue | 102 | 2 |

**Explanation:** Order 1001 belongs to customer 1, whose `full_name` "John Doe" splits into John / Doe, and product 101, whose `product_info` "Carpet,Red" splits into Carpet / Red.

### Constraints

- Return the columns in exactly this order: `customer_id`, `first_name`, `last_name`, `location`, `order_id`, `product_type`, `product_color`, `product_id`, `quantity`.
- Split `full_name` on the single space; split `product_info` on the comma.
- Use inner joins, so only orders with a matching customer and product appear.
- Return rows in ascending order of `order_id`.

In [0]:
fr_customers_data = [(1,"John Doe","Texas"),(2,"Jane Smith","California")]
fr_customers_df = spark.createDataFrame(fr_customers_data, ["customer_id","full_name","location"])

fr_orders_data = [(1001,1,101,5),(1002,2,102,2)]
fr_orders_df = spark.createDataFrame(fr_orders_data, ["order_id","customer_id","product_id","quantity"])

fr_products_data = [(101,"Carpet,Red"),(102,"Tile,Blue")]
fr_products_df = spark.createDataFrame(fr_products_data, ["product_id","product_info"])

# display(fr_customers_df)
# display(fr_orders_df)
# display(fr_products_df)

customer_df = (
fr_customers_df
    .withColumn("name_list", split(col("full_name"), " "))
    .withColumn("first_name", col("name_list")[0])
    .withColumn("last_name", col("name_list")[1])
    .drop("name_list")
)

product_df = (
fr_products_df
    .withColumn("info_list", split(col("product_info"), ","))
    .withColumn("product_type", col("info_list")[0])
    .withColumn("color", col("info_list")[1])
    .drop("info_list")
)

display(customer_df)
display(product_df)
display(fr_orders_df)

# customer_id	first_name	last_name	location	order_id	product_type	product_color	product_id	quantity

joined_df = (
fr_orders_df.alias("o")
    .join(customer_df.alias("c"), col("o.customer_id") == col("c.customer_id"))
    .join(product_df.alias("p"), col("o.product_id") == col("p.product_id"))
    .select("c.customer_id", "c.first_name", "c.last_name", "c.location", "o.order_id", "p.product_type", "p.color", "o.product_id", "o.quantity")
)

display(joined_df)
